# Preprocessing

On part des CSV bruts sur S3 et on ressort un dataset propre, prêt pour le feature engineering.
Filtre sur 1990+, harmonisation des noms de pays, et création de la variable cible.

In [2]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# On charge les credentials AWS depuis le .env
print("Chargement des variables d'environnement...")
load_dotenv("../.env")
print("...fait")

# Chemins S3 vers les données brutes et les données traitées
BUCKET = "fifa-wc-predict-915328198414-eu-west-3-an"
RAW    = f"s3://{BUCKET}/raw"
PROC   = f"s3://{BUCKET}/processed"

Chargement des variables d'environnement...
...fait


## Chargement depuis S3

In [6]:
# On charge les 4 fichiers bruts directement depuis S3
print("Chargement des fichiers depuis S3...")
results      = pd.read_csv(f"{RAW}/results.csv", parse_dates=['date'])
goals  = pd.read_csv(f"{RAW}/goalscorers.csv", parse_dates=['date'])
shootouts    = pd.read_csv(f"{RAW}/shootouts.csv", parse_dates=['date'])
former = pd.read_csv(f"{RAW}/former_names.csv")
print("...fait")

# On vérifie que tout est bien chargé
print(f"results      : {results.shape}")
print(f"goals  : {goals.shape}")
print(f"shootouts    : {shootouts.shape}")
print(f"former : {former.shape}")

Chargement des fichiers depuis S3...
...fait
results      : (49287, 9)
goals  : (47601, 8)
shootouts    : (675, 5)
former : (36, 4)


## Exploration rapide

In [7]:
# On jette un coup d'oeil aux premières lignes
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [8]:
# On vérifie les types de colonnes et les éventuels nulls
results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49287 entries, 0 to 49286
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        49287 non-null  datetime64[ns]
 1   home_team   49287 non-null  object        
 2   away_team   49287 non-null  object        
 3   home_score  49215 non-null  float64       
 4   away_score  49215 non-null  float64       
 5   tournament  49287 non-null  object        
 6   city        49287 non-null  object        
 7   country     49287 non-null  object        
 8   neutral     49287 non-null  bool          
dtypes: bool(1), datetime64[ns](1), float64(2), object(5)
memory usage: 3.1+ MB


In [9]:
# On regarde quels tournois sont représentés dans le dataset
results["tournament"].value_counts().head(20)

tournament
Friendly                                18252
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Gold Cup                                  420
Gulf Cup                                  410
Island Games                              394
UEFA Euro                                 388
Asian Games                               368
Name: count, dtype: int64

## Filtre 1990+

On garde uniquement les matchs depuis 1990 — les données plus anciennes sont trop hétérogènes
et peu représentatives du football actuel.

In [ ]:
# On ne garde que les matchs à partir du 1er janvier 1990
print("Filtrage sur 1990+...")
df = results[results["date"] >= "1990-01-01"].copy()
print("...fait")

print(f"Avant  : {len(results):,} matchs")
print(f"Après  : {len(df):,} matchs")
print(f"Période: {df['date'].min().date()} → {df['date'].max().date()}")

## Harmonisation des noms de pays

Certains pays ont changé de nom (ex: Zaïre → DR Congo, Yougoslavie → Serbie...).
On remplace les anciens noms par les noms actuels pour éviter de traiter la même équipe comme deux équipes différentes.

In [12]:
# On regarde à quoi ressemble le fichier des anciens noms
former.head(10)

,current,former,start_date,end_date
0,Benin,Dahomey,1959-11-08,1975-11-30
1,Burkina Faso,Upper Volta,1960-04-14,1984-08-04
2,Curaçao,Netherlands Antilles,1957-03-03,2010-10-10
3,Czechoslovakia,Bohemia,1903-04-05,1919-01-01
4,Czechoslovakia,Bohemia and Moravia,1939-01-01,1945-05-01
5,Czechoslovakia,Representation of Czechs and Slovaks,1993-03-24,1993-11-17
6,DR Congo,Belgian Congo,1948-05-25,1956-01-02
7,DR Congo,Congo-Léopoldville,1963-04-12,1964-07-19
8,DR Congo,Congo-Kinshasa,1965-01-09,1970-11-24
9,DR Congo,Zaïre,1971-01-10,1997-04-27


In [15]:
# On construit un dictionnaire ancien nom -> nom actuel
print("Harmonisation des noms de pays...")
name_mapping = dict(zip(former["former"], former["current"]))

# On remplace dans results
df["home_team"] = df["home_team"].replace(name_mapping)
df["away_team"] = df["away_team"].replace(name_mapping)

# On harmonise aussi goalscorers, on en aura besoin pour les features
goals["home_team"] = goals["home_team"].replace(name_mapping)
goals["away_team"] = goals["away_team"].replace(name_mapping)
goals["team"]      = goals["team"].replace(name_mapping)
print("...fait")

print(f"{len(name_mapping)} remplacements appliqués")

Harmonisation des noms de pays...
...fait
36 remplacements appliqués


## Variable cible

On crée la colonne `result` :
- `2` → victoire domicile
- `1` → nul
- `0` → défaite domicile

In [16]:
# On encode le résultat de chaque match selon les scores
def encode_result(row):
    if row["home_score"] > row["away_score"]:
        return 2  # victoire domicile
    elif row["home_score"] == row["away_score"]:
        return 1  # nul
    else:
        return 0  # défaite domicile

print("Encodage de la variable cible...")
df["result"] = df.apply(encode_result, axis=1)
print("...fait")

# On vérifie la répartition des 3 classes
counts = df["result"].value_counts().sort_index()
labels = {0: "Défaite domicile", 1: "Nul", 2: "Victoire domicile"}

for code, count in counts.items():
    pct = count / len(df) * 100
    print(f"{labels[code]:25s} ({code}) : {count:,} matchs ({pct:.1f}%)")

Encodage de la variable cible...
...fait
Défaite domicile          (0) : 7,264 matchs (28.8%)
Nul                       (1) : 5,867 matchs (23.3%)
Victoire domicile         (2) : 12,098 matchs (48.0%)


## Valeurs manquantes

In [17]:
# On s'assure qu'il ne reste pas de valeurs manquantes avant de sauvegarder
print("Vérification des valeurs manquantes...")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "...aucune valeur manquante")

Vérification des valeurs manquantes...
home_score    72
away_score    72
dtype: int64


In [ ]:
# Aperçu du dataset final
print(f"Dataset final : {df.shape[0]:,} matchs — {df.shape[1]} colonnes")
df.head()

## Sauvegarde sur S3

In [ ]:
# On sauvegarde results nettoyé dans processed/
print("Sauvegarde de results_clean.csv...")
df.to_csv(f"{PROC}/results_clean.csv", index=False)
print(f"...fait : {PROC}/results_clean.csv")

# On filtre et sauvegarde goalscorers également, on en aura besoin pour les features
print("Sauvegarde de goalscorers_clean.csv...")
goalscorers_filtered = goalscorers[
    pd.to_datetime(goalscorers["date"]) >= "1990-01-01"
].copy()
goalscorers_filtered.to_csv(f"{PROC}/goalscorers_clean.csv", index=False)
print(f"...fait : {PROC}/goalscorers_clean.csv")